<!-- # ReAct Agent -->

In [2]:
from typing import Annotated, Sequence, TypedDict
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage
from langchain_core.messages import ToolMessage
from langchain_core.messages import SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END, START
from langgraph.prebuilt import ToolNode

load_dotenv()  # take environment variables from .env file

True

In [3]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

In [17]:
@tool
def add(a: int, b: int):
    """ Add two numbers. """
    return a + b

@tool
def substract(a: int, b: int):
    """ Substract numbers """
    return a - b

@tool
def multiply(a: int, b: int):
    """ multiply numbers """
    return a * b

@tool
def divide(a: int, b: int):
    """ divide numbers """
    if b == 0:
        raise ValueError("Division by zero is not allowed")
    return a / b



In [18]:
tools = [add, substract, multiply, divide]
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash").bind_tools(tools)


In [27]:
def model_call(state: AgentState) -> AgentState:
    system_prompt = SystemMessage(content="You are a helpful assistant, answer my question with best quality, return result as 'Result':[result]")
    response = model.invoke([system_prompt] + state["messages"])
    return {"messages":[response]}

In [28]:
def should_continue(state: AgentState) -> str:
    messages = state['messages']
    last_message = messages[-1]
    if not last_message.tool_calls:
        return "end"
    else:
        return "continue"

In [29]:
graph = StateGraph(AgentState)
graph.add_node("our_agent", model_call)

tool_node = ToolNode(tools=tools)
graph.add_node("tools", tool_node)

graph.add_edge(START, "our_agent")

graph.add_conditional_edges(
    "our_agent",
    should_continue,
    {
        "continue": "tools",
        "end": END
    }
)

graph.add_edge("tools", "our_agent")

In [30]:
app = graph.compile()

In [31]:
def print_stream(stream):
    for s in stream:
        message = s['messages'][-1]
        if isinstance(message, tuple):
            print(message)
        else:
            message.pretty_print()

inputs = {"messages": [{"role": "user", "content": "5 + 6 and take the result and multiply it with 2"}]}
print_stream(app.stream(inputs, stream_mode="values"))

================================ Human Message =================================

5 + 6 and take the result and multiply it with 2
================================== Ai Message ==================================
Tool Calls:
  add (e1ae3de7-783b-49eb-90fa-4bee22cee1bc)
 Call ID: e1ae3de7-783b-49eb-90fa-4bee22cee1bc
  Args:
    a: 5
    b: 6
================================= Tool Message =================================
Name: add

11
================================== Ai Message ==================================
Tool Calls:
  multiply (40f6930f-b9f3-4112-a0ee-043b0ff8cc16)
 Call ID: 40f6930f-b9f3-4112-a0ee-043b0ff8cc16
  Args:
    a: 11
    b: 2
================================= Tool Message =================================
Name: multiply

22
================================== Ai Message ==================================

Result: 22


In [11]:
# 